<div style="background: linear-gradient(135deg, #1a1a2e 0%, #16213e 50%, #0f3460 100%); padding: 40px 30px; border-radius: 12px; margin-bottom: 10px;">
    <h1 style="color:#e2b96f; font-family:'Segoe UI', sans-serif; font-size:2.2em; margin:0 0 8px 0;">
        🎓 Introdução ao Aprendizado de Máquina
    </h1>
    <h2 style="color:#a8d8ea; font-family:'Segoe UI', sans-serif; font-size:1.3em; margin:0 0 6px 0; font-weight:400;">
        Aula 08 — Árvore de Decisão
    </h2>
    <h3 style="color:#e2b96f; font-family:'Segoe UI', sans-serif; font-size:1.05em; margin:0 0 12px 0; font-weight:500;">
        🔬 Prática — O Modelo que Explica Cada Decisão no Titanic
    </h3>
    <p style="color:#ccc; font-family:'Segoe UI', sans-serif; font-size:0.9em; margin:0;">
        Prof. Felipe Amaral
    </p>
</div>
<div style="display:flex; gap:10px; margin-top:10px; flex-wrap:wrap;">
    <span style="background:#0f3460; color:#a8d8ea; padding:5px 14px; border-radius:20px; font-size:0.85em;">📚 FIAP</span>
    <span style="background:#0f3460; color:#a8d8ea; padding:5px 14px; border-radius:20px; font-size:0.85em;">🐍 Python 3</span>
    <span style="background:#0f3460; color:#a8d8ea; padding:5px 14px; border-radius:20px; font-size:0.85em;">🚢 Dataset Titanic</span>
    <span style="background:#0f3460; color:#a8d8ea; padding:5px 14px; border-radius:20px; font-size:0.85em;">🌳 5º Modelo do Curso</span>
</div>


## Onde estamos?

| Aula | Modelo | Ideia central | Explica a decisão? |
|------|--------|---------------|-------------------|
| 04 | Reg. Linear | Prever um número via reta | Sim — coeficientes |
| 05 | Reg. Logística | Probabilidade via sigmoide | Sim — coeficientes |
| 06 | KNN | Votação dos K vizinhos mais próximos | Não |
| 07 | SVM | Hiperplano de margem máxima | Parcialmente |
| **08** | **Árvore de Decisão** | **Perguntas encadeadas sobre as features** | **Sim — nó a nó** |

Hoje treinamos o modelo **mais intuitivo** do curso. A Árvore de Decisão faz
algo que qualquer pessoa entende:

> *"É mulher? → Sim: provavelmente sobreviveu.*
> *Não: está na 1ª ou 2ª classe? → Sim: alguma chance. Não: provavelmente não sobreviveu."*

Isso é exatamente o que a árvore aprende — e você consegue **mostrar para
alguém, passo a passo, como cada decisão foi tomada**.

### Roteiro de hoje

| Parte | Tema |
|-------|------|
| **1** | Intuição — como a árvore faz perguntas |
| **2** | Gini — como o algoritmo escolhe a melhor pergunta |
| **3** | Treinando e visualizando a árvore no Titanic |
| **4** | Overfitting e poda — controlando a profundidade |
| **5** | Feature Importance e placar final |

> **Tempo estimado: 35 minutos**

<div style="background:#d4edda; border-left:5px solid #155724; padding:14px 20px; border-radius:6px; margin:12px 0;">
<strong style="color:#155724;">Por que aprender a Árvore agora?</strong>
<span style="color:#155724;"> Ela é a base de algoritmos muito mais poderosos que veremos a seguir: <strong>Random Forest</strong> e <strong>Gradient Boosting (XGBoost)</strong>. Dominar a árvore individual é o pré-requisito essencial.</span>
</div>


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings("ignore")

plt.rcParams.update({
    "figure.figsize":  (10, 5),
    "axes.spines.top":    False,
    "axes.spines.right":  False,
    "axes.titlesize":     13,
    "axes.labelsize":     11,
})
sns.set_theme(style="whitegrid", palette="muted")

from sklearn.model_selection import train_test_split

# ── Carregando e preparando o Titanic (mesma limpeza das aulas anteriores) ────
df = sns.load_dataset("titanic").copy()

df["age"]      = df["age"].fillna(df["age"].median())
df["embarked"] = df["embarked"].fillna(df["embarked"].mode()[0])
df = df.drop(columns=["deck"]).drop_duplicates().reset_index(drop=True)

df["tamanho_familia"] = df["sibsp"] + df["parch"] + 1
df["sex_enc"]         = (df["sex"] == "female").astype(int)

# 5 features simples e fáceis de explicar
FEATURES = ["pclass", "sex_enc", "age", "tamanho_familia", "fare"]

X = df[FEATURES].copy()
y = df["survived"].copy()

X_treino, X_teste, y_treino, y_teste = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

print("✅ Dataset pronto!")
print(f"   Treino: {len(X_treino)} passageiros  |  Teste: {len(X_teste)} passageiros")
print(f"   Features usadas: {FEATURES}")
print()
print("Nota importante: a Árvore de Decisão NÃO precisa de normalização —")
print("ela usa pontos de corte, não distâncias. Vamos treinar com os dados brutos.")


---

<div style="background: linear-gradient(90deg, #0f3460 0%, #16213e 100%); padding:20px 25px; border-radius:10px; margin:20px 0;">
    <div>
        <span style="color:#e2b96f; font-size:0.85em; font-weight:bold; letter-spacing:2px;">PARTE 1</span>
        <h2 style="color:white; margin:4px 0 0 0; font-family:'Segoe UI',sans-serif;">Intuição — Como uma Árvore Toma Decisões</h2>
        <p style="color:#a8d8ea; margin:6px 0 0 0; font-size:0.9em;">"O modelo que qualquer pessoa consegue ler e entender."</p>
    </div>
</div>

### A ideia central

Uma Árvore de Decisão aprende uma sequência de **perguntas binárias** sobre as
features. Em cada nó, ela pergunta: *"o valor desta feature é maior ou menor
que X?"* Dependendo da resposta, o exemplo vai para o galho esquerdo ou
direito. No final, chega a uma **folha** — que contém a classe prevista.

```
                    [Nó Raiz]
                  sex_enc <= 0.5?
                 /               \
          Sim (homem)          Não (mulher)
              /                      \
       [Nó Interno]              [Folha]
    pclass <= 1.5?            Prevê: SOBREVIVEU
       /         \
   Sim            Não
 [Folha]         [Folha]
Prevê: NÃO      Prevê: NÃO
```

### Vocabulário da árvore

| Termo | Significado |
|-------|-------------|
| **Nó Raiz** | A primeira pergunta — a feature mais informativa |
| **Nó Interno** | Uma pergunta que divide os dados em dois grupos |
| **Folha** | Nó terminal — contém a previsão final |
| **Profundidade** | Número de perguntas do raiz até a folha mais distante |

### A grande vantagem: explicabilidade

O KNN disse "os vizinhos mais próximos votaram assim". O SVM disse "o ponto
está do lado correto do hiperplano". A Árvore diz: **"passei por estas
perguntas e cheguei nesta resposta"** — um modelo **caixa-branca**, auditável
passo a passo.


<div style="background:#fde8d8; border-left:5px solid #a04000; padding:14px 20px; border-radius:6px; margin:12px 0;"><strong style="color:#a04000;">🎯 </strong><span style="color:#a04000;">MISSÃO 1 — Antes de rodar qualquer código, monte manualmente uma árvore simples de profundidade 2 para o Titanic. Escreva as duas perguntas que você usaria e justifique.</span></div>

*✏️ Minha árvore manual (profundidade 2):*

```
Pergunta 1 (Raiz): ???
    Sim → Pergunta 2a: ???
        Sim → Prevê: ???
        Não → Prevê: ???
    Não → Prevê: ???
```

*Justificativa: `???`*


In [ ]:
# Confirmando na prática: qual é a melhor primeira pergunta?
# A taxa de sobrevivência por grupo nos dá uma pista
print("Taxa de sobrevivência por grupo — ajuda a montar a árvore manual")

grupos = {
    "sex_enc == 1 (mulher)": df["sex_enc"] == 1,
    "pclass == 1 (1ª classe)": df["pclass"] == 1,
    "tamanho_familia == 1 (sozinho)": df["tamanho_familia"] == 1,
}

for nome, mask in grupos.items():
    taxa_sim = df.loc[mask, "survived"].mean()
    taxa_nao = df.loc[~mask, "survived"].mean()
    print(f"  {nome:<32}  Sim: {taxa_sim:.0%}  Não: {taxa_nao:.0%}  "
          f"Separação: {abs(taxa_sim-taxa_nao):.0%}")

print()
print("Quanto maior a 'Separação', melhor essa feature como primeira pergunta!")


In [ ]:
# ── GABARITO DA MISSÃO 1 (descomente para ver) ───────────────────────────────
# print("Pergunta 1 (Raiz): sex_enc <= 0.5?  (é homem?)")
# print("  Sim (homem):")
# print("    Pergunta 2a: pclass <= 1.5?  (é 1ª classe?)")
# print("      Sim (1ª classe) -> Prevê: NÃO SOBREVIVEU  (mas taxa maior que 3ª)")
# print("      Não (2ª/3ª)     -> Prevê: NÃO SOBREVIVEU  (taxa histórica baixa)")
# print("  Não (mulher):")
# print("    -> Prevê: SOBREVIVEU  (taxa histórica alta)")
# print()
# print("Justificativa: sex_enc tem a maior separação — mulheres sobreviveram")
# print("muito mais. É exatamente o que a árvore treinada vai aprender!")


---

<div style="background: linear-gradient(90deg, #0f3460 0%, #16213e 100%); padding:20px 25px; border-radius:10px; margin:20px 0;">
    <div>
        <span style="color:#e2b96f; font-size:0.85em; font-weight:bold; letter-spacing:2px;">PARTE 2</span>
        <h2 style="color:white; margin:4px 0 0 0; font-family:'Segoe UI',sans-serif;">Gini — Como Escolher a Melhor Pergunta</h2>
        <p style="color:#a8d8ea; margin:6px 0 0 0; font-size:0.9em;">"O algoritmo precisa de um critério matemático para comparar perguntas."</p>
    </div>
</div>

### O problema da escolha

A árvore pode fazer milhares de perguntas diferentes. Para escolher a
**melhor em cada nó**, ela usa uma medida de **impureza**: o quão misturado
está o grupo resultante após a divisão.

> Um grupo totalmente puro tem só uma classe — o ideal. Um grupo 50/50 é
> completamente impuro — a pior situação.

### Impureza de Gini

```
Gini(nó) = 1 − Σ pᵢ²    onde pᵢ = proporção de exemplos da classe i no nó
```

| Situação | Gini |
|----------|------|
| 100% de uma classe | **0.0** — totalmente puro |
| 50% / 50% | **0.5** — máxima impureza |
| 70% / 30% | 1 − (0.7² + 0.3²) = **0.42** |

### Ganho de Gini

A árvore escolhe a pergunta que **mais reduz a impureza** nos grupos filhos:

```
Ganho = Gini(pai) − Σ [ (nᵢ/n) × Gini(filho i) ]
```

Quanto maior o ganho, melhor a pergunta.


In [ ]:
# Calculando Gini manualmente para uma divisão do Titanic
total      = len(df)
sobrev     = df["survived"].sum()
p_sobrev   = sobrev / total
p_nao      = 1 - p_sobrev

gini_raiz = 1 - (p_sobrev**2 + p_nao**2)
print(f"Nó Raiz (todos os {total} passageiros):")
print(f"  Sobreviveram: {p_sobrev:.1%}  |  Não sobrev.: {p_nao:.1%}")
print(f"  Gini(raiz) = 1 - ({p_sobrev:.3f}² + {p_nao:.3f}²) = {gini_raiz:.4f}")
print()

def gini_no(grp):
    p = grp["survived"].mean()
    return 1 - (p**2 + (1-p)**2)

# Dividindo por sex_enc (0=homem, 1=mulher)
grp_h = df[df["sex_enc"] == 0]
grp_m = df[df["sex_enc"] == 1]

print("Divisão por sex_enc:")
for nome, grp in [("Homens", grp_h), ("Mulheres", grp_m)]:
    print(f"  {nome:<10}: n={len(grp):>3}  sobrev={grp['survived'].mean():.0%}  "
          f"Gini={gini_no(grp):.4f}")

gini_pond_sexo = (len(grp_h)/total)*gini_no(grp_h) + (len(grp_m)/total)*gini_no(grp_m)
ganho_sexo = gini_raiz - gini_pond_sexo

print()
print(f"Gini ponderado após dividir por sexo: {gini_pond_sexo:.4f}")
print(f"Ganho de Gini: {gini_raiz:.4f} - {gini_pond_sexo:.4f} = {ganho_sexo:.4f}")


<div style="background:#fde8d8; border-left:5px solid #a04000; padding:14px 20px; border-radius:6px; margin:12px 0;"><strong style="color:#a04000;">🎯 </strong><span style="color:#a04000;">MISSÃO 2 — Calcule o Ganho de Gini para a divisão por <code>pclass</code> (separando 1ª classe das demais, corte em 1.5) e compare com o ganho por <code>sex_enc</code>. Qual feature a árvore escolheria como raiz?</span></div>

In [ ]:
# ✏️ MISSÃO 2 — complete o código
grp_1a  = df[df["pclass"] <= 1.5]
grp_23a = df[df["pclass"] > 1.5]

# ✏️ Calcule o Gini ponderado e o ganho para essa divisão
# gini_pond_pclass = ???
# ganho_pclass     = ???

# print(f"Ganho por sex_enc: {ganho_sexo:.4f}")
# print(f"Ganho por pclass:  {ganho_pclass:.4f}")
# print(f"Melhor divisão: {'sex_enc' if ganho_sexo > ganho_pclass else 'pclass'}")


In [ ]:
# ── GABARITO DA MISSÃO 2 (descomente para ver) ───────────────────────────────
# gini_pond_pclass = ((len(grp_1a)/total)*gini_no(grp_1a) +
#                     (len(grp_23a)/total)*gini_no(grp_23a))
# ganho_pclass = gini_raiz - gini_pond_pclass
#
# print(f"Ganho por sex_enc: {ganho_sexo:.4f}")
# print(f"Ganho por pclass:  {ganho_pclass:.4f}")
# print()
# if ganho_sexo > ganho_pclass:
#     print("Vencedor: sex_enc — a árvore usará essa como raiz!")
#     print("Faz sentido: a separação por gênero é mais limpa que por classe.")
# else:
#     print("Vencedor: pclass")


---

<div style="background: linear-gradient(90deg, #0f3460 0%, #16213e 100%); padding:20px 25px; border-radius:10px; margin:20px 0;">
    <div>
        <span style="color:#e2b96f; font-size:0.85em; font-weight:bold; letter-spacing:2px;">PARTE 3</span>
        <h2 style="color:white; margin:4px 0 0 0; font-family:'Segoe UI',sans-serif;">Treinando e Visualizando a Árvore</h2>
        <p style="color:#a8d8ea; margin:6px 0 0 0; font-size:0.9em;">"A grande vantagem: você pode mostrar a árvore para qualquer pessoa."</p>
    </div>
</div>

O `scikit-learn` usa o algoritmo **CART**, que repete o processo da Parte 2 em
cada nó: testa todas as features e pontos de corte possíveis, escolhe o de
maior Ganho de Gini, e repete recursivamente nos filhos até atingir a
profundidade máxima ou um nó já puro.

Vamos treinar com profundidade limitada a 3, boa para visualizar.


In [ ]:
from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text

arvore_viz = DecisionTreeClassifier(max_depth=3, criterion="gini", random_state=42)
arvore_viz.fit(X_treino, y_treino)

print("Árvore treinada (profundidade=3)!")
print(f"  Nós folha:         {arvore_viz.get_n_leaves()}")
print(f"  Acurácia treino:   {arvore_viz.score(X_treino, y_treino):.1%}")
print(f"  Acurácia teste:    {arvore_viz.score(X_teste,  y_teste):.1%}")


In [ ]:
# Visualizando a árvore — o grande diferencial do algoritmo
fig, ax = plt.subplots(figsize=(18, 8))
plot_tree(arvore_viz, feature_names=FEATURES,
          class_names=["Não Sobrev.", "Sobreviveu"],
          filled=True, rounded=True, impurity=True, fontsize=9, ax=ax)
ax.set_title("Árvore de Decisão — Titanic (profundidade=3)\n"
             "Azul = predomina 'Sobreviveu' | Laranja = predomina 'Não Sobreviveu'",
             fontsize=11, fontweight="bold")
plt.tight_layout()
plt.show()


In [ ]:
# Rastreando a decisão para um passageiro específico
idx = 10
passageiro = X_teste.values[idx]
real       = y_teste.values[idx]

print(f"Passageiro #{idx} — features: {dict(zip(FEATURES, passageiro))}")
print()

node_indicator = arvore_viz.decision_path(passageiro.reshape(1, -1))
leaf_id        = arvore_viz.apply(passageiro.reshape(1, -1))
threshold      = arvore_viz.tree_.threshold
feature_arr    = arvore_viz.tree_.feature

print("Caminho percorrido na árvore:")
for node_id in node_indicator.indices:
    if leaf_id[0] == node_id:
        previsao = arvore_viz.predict(passageiro.reshape(1, -1))[0]
        print(f"  → FOLHA: Previsão = {'Sobreviveu' if previsao==1 else 'Não Sobreviveu'}")
    else:
        feat_nome = FEATURES[feature_arr[node_id]]
        corte     = threshold[node_id]
        val       = passageiro[feature_arr[node_id]]
        direcao   = "SIM, vai para esquerda" if val <= corte else "NÃO, vai para direita"
        print(f"  Nó {node_id}: {feat_nome} <= {corte:.2f}? "
              f"(valor real = {val:.2f}) → {direcao}")

print()
print(f"Resultado real: {'Sobreviveu ✅' if real==1 else 'Não Sobreviveu ❌'}")


<div style="background:#fde8d8; border-left:5px solid #a04000; padding:14px 20px; border-radius:6px; margin:12px 0;"><strong style="color:#a04000;">🎯 </strong><span style="color:#a04000;">MISSÃO 3 — Olhando a árvore visualizada: (a) qual foi a primeira pergunta (nó raiz)? Corresponde à sua hipótese da Missão 1? (b) existe algum nó com Gini=0.0? O que isso significa?</span></div>

*✏️ (a) Primeiro nó (raiz): `???` — corresponde à minha hipótese? `???`*

*✏️ (b) Existe nó com Gini=0? `???` — isso significa: `???`*


In [ ]:
# ── GABARITO DA MISSÃO 3 (descomente para ver) ───────────────────────────────
# print(f"(a) Primeiro nó (raiz): {FEATURES[arvore_viz.tree_.feature[0]]}"
#       f" <= {arvore_viz.tree_.threshold[0]:.2f}")
# print("    Deve corresponder à Missão 1 — sex_enc costuma ser a melhor divisão.")
# print()
# print("(b) Nós com Gini=0.0 são completamente puros — todos os exemplos são")
# print("    da mesma classe. Em dados de treino, muitos nós puros podem indicar")
# print("    overfitting — veremos isso na Parte 4.")


---

<div style="background: linear-gradient(90deg, #0f3460 0%, #16213e 100%); padding:20px 25px; border-radius:10px; margin:20px 0;">
    <div>
        <span style="color:#e2b96f; font-size:0.85em; font-weight:bold; letter-spacing:2px;">PARTE 4</span>
        <h2 style="color:white; margin:4px 0 0 0; font-family:'Segoe UI',sans-serif;">Overfitting e Poda — Controlando a Profundidade</h2>
        <p style="color:#a8d8ea; margin:6px 0 0 0; font-size:0.9em;">"Uma árvore muito funda decora o treino. Uma muito rasa ignora os padrões."</p>
    </div>
</div>

Uma árvore sem limitação de profundidade pode crescer até ter **uma folha por
exemplo** — 100% de acerto no treino e péssimo desempenho no teste.

```
Profundidade 1:  muito simples → underfitting
Profundidade 3-5: balanceado  → bom para generalizar
Profundidade 20+: muito fundo → overfitting (decora ruído)
```

O parâmetro mais importante para controlar isso é `max_depth`.


In [ ]:
# Curva de profundidade: treino vs teste
profundidades = range(1, 16)
accs_treino   = []
accs_teste    = []

for d in profundidades:
    arvore_d = DecisionTreeClassifier(max_depth=d, criterion="gini", random_state=42)
    arvore_d.fit(X_treino, y_treino)
    accs_treino.append(arvore_d.score(X_treino, y_treino))
    accs_teste.append(arvore_d.score(X_teste, y_teste))

melhor_d = list(profundidades)[np.argmax(accs_teste)]

plt.figure(figsize=(10, 5))
plt.plot(profundidades, [a*100 for a in accs_treino], "o--", color="#e94560", label="Treino")
plt.plot(profundidades, [a*100 for a in accs_teste],  "s-",  color="#0f3460", label="Teste")
plt.axvline(melhor_d, color="#2ecc71", linestyle="--",
            label=f"Melhor profundidade (teste) = {melhor_d}")
plt.xlabel("Profundidade Máxima"); plt.ylabel("Acurácia (%)")
plt.title("Árvore de Decisão — Overfitting vs Underfitting", fontweight="bold")
plt.legend()
plt.tight_layout()
plt.show()

print(f"Profundidade {melhor_d}: treino={accs_treino[melhor_d-1]:.1%}  "
      f"teste={accs_teste[melhor_d-1]:.1%}")


<div style="background:#fde8d8; border-left:5px solid #a04000; padding:14px 20px; border-radius:6px; margin:12px 0;"><strong style="color:#a04000;">🎯 </strong><span style="color:#a04000;">MISSÃO 4 — Olhando a curva: (a) a partir de qual profundidade a linha de treino continua subindo enquanto a de teste cai (sinal de overfitting)? (b) por que não escolhemos simplesmente a profundidade com maior acurácia de TREINO?</span></div>

*✏️ (a) Overfitting começa por volta da profundidade: `???`*

*✏️ (b) Não uso a acurácia de treino para escolher porque: `???`*


In [ ]:
# ── GABARITO DA MISSÃO 4 (descomente para ver) ───────────────────────────────
# print("(a) A acurácia de treino sobe (quase) sempre com a profundidade —")
# print("    é a de TESTE que revela o overfitting quando começa a cair ou estagnar.")
# print()
# print("(b) A acurácia de treino sempre pode chegar perto de 100% com profundidade")
# print("    alta o suficiente — isso não significa que o modelo generaliza bem.")
# print("    Ele pode estar decorando ruído específico dos dados de treino.")


In [ ]:
# Treinando a árvore final com a melhor profundidade encontrada
arvore_final = DecisionTreeClassifier(
    max_depth=melhor_d, criterion="gini", min_samples_leaf=5, random_state=42)
arvore_final.fit(X_treino, y_treino)

from sklearn.metrics import classification_report
y_pred_arvore = arvore_final.predict(X_teste)

print(f"ÁRVORE FINAL (max_depth={melhor_d}, min_samples_leaf=5)")
print(classification_report(y_teste, y_pred_arvore,
      target_names=["Não Sobreviveu", "Sobreviveu"]))


---

<div style="background: linear-gradient(90deg, #0f3460 0%, #16213e 100%); padding:20px 25px; border-radius:10px; margin:20px 0;">
    <div>
        <span style="color:#e2b96f; font-size:0.85em; font-weight:bold; letter-spacing:2px;">PARTE 5</span>
        <h2 style="color:white; margin:4px 0 0 0; font-family:'Segoe UI',sans-serif;">Feature Importance e Placar Final</h2>
        <p style="color:#a8d8ea; margin:6px 0 0 0; font-size:0.9em;">"A árvore nos diz não só o que previu, mas quais features mais importaram."</p>
    </div>
</div>

### Feature Importance

A importância de cada feature é calculada pela **redução total de Gini** que
ela proporcionou ao longo de todas as divisões da árvore. Quanto mais vezes
uma feature é usada e quanto maior o ganho que traz, maior sua importância.


In [ ]:
# Feature importance da árvore final
importancias = pd.DataFrame({
    "feature":     FEATURES,
    "importancia": arvore_final.feature_importances_
}).sort_values("importancia", ascending=False)

plt.figure(figsize=(8, 4.5))
plt.barh(importancias["feature"], importancias["importancia"], color="#0f3460", edgecolor="white")
plt.xlabel("Importância (redução total de Gini)")
plt.title("Feature Importance — Árvore de Decisão", fontweight="bold")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

print(importancias.round(4).to_string(index=False))


### Comparando com os modelos anteriores

Vamos treinar rapidamente o KNN e a Regressão Logística (das Aulas 05 e 06)
para fechar o curso com um placar final entre os três.


In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, f1_score, precision_score,
                              recall_score, roc_auc_score, roc_curve)

scaler = StandardScaler()
X_treino_sc = scaler.fit_transform(X_treino)
X_teste_sc  = scaler.transform(X_teste)

knn_ref = KNeighborsClassifier(n_neighbors=7).fit(X_treino_sc, y_treino)
lr_ref  = LogisticRegression(max_iter=1000, random_state=42).fit(X_treino_sc, y_treino)

modelos = {
    "KNN (K=7)":            (knn_ref,      X_teste_sc),
    "Reg. Logística":       (lr_ref,       X_teste_sc),
    "Árvore (otimizada)":   (arvore_final, X_teste),
}

tabela = []
for nome, (modelo, Xte) in modelos.items():
    yp  = modelo.predict(Xte)
    ypr = modelo.predict_proba(Xte)[:, 1]
    tabela.append({
        "Modelo":   nome,
        "Acurácia": accuracy_score(y_teste, yp),
        "Precisão": precision_score(y_teste, yp),
        "Recall":   recall_score(y_teste, yp),
        "F1":       f1_score(y_teste, yp),
        "AUC-ROC":  roc_auc_score(y_teste, ypr),
    })

df_tabela = pd.DataFrame(tabela).set_index("Modelo")
print("PLACAR FINAL — KNN vs Reg. Logística vs Árvore de Decisão")
print(df_tabela.round(4).to_string())


In [ ]:
# Curvas ROC sobrepostas
plt.figure(figsize=(7, 6))
cores = ["#a8d8ea", "#0f3460", "#e94560"]
for cor, (nome, (modelo, Xte)) in zip(cores, modelos.items()):
    ypr = modelo.predict_proba(Xte)[:, 1]
    fpr, tpr, _ = roc_curve(y_teste, ypr)
    auc = roc_auc_score(y_teste, ypr)
    plt.plot(fpr, tpr, color=cor, linewidth=2.2, label=f"{nome} (AUC={auc:.3f})")

plt.plot([0,1], [0,1], "k--", linewidth=1, alpha=0.4, label="Aleatório")
plt.xlabel("Taxa de Falso Positivo")
plt.ylabel("Taxa de Verdadeiro Positivo (Recall)")
plt.title("Curvas ROC — Comparação Final", fontweight="bold")
plt.legend()
plt.tight_layout()
plt.show()


<div style="background:#fde8d8; border-left:5px solid #a04000; padding:14px 20px; border-radius:6px; margin:12px 0;"><strong style="color:#a04000;">🎯 </strong><span style="color:#a04000;">MISSÃO 5 (final) — Compare a Feature Importance da Árvore com os coeficientes da Regressão Logística (Aula 05): (a) as top 2 features concordam entre os dois modelos? (b) qual modelo você escolheria se precisasse explicar a decisão para um passageiro?</span></div>

*✏️ (a) Top 2 concordam? `???`*

*✏️ (b) Escolheria: `???` para explicar a um passageiro, porque: `???`*


In [ ]:
# ── GABARITO DA MISSÃO 5 (descomente para ver) ───────────────────────────────
# top_arvore = importancias.head(2)["feature"].tolist()
# top_lr = (pd.DataFrame({"feature": FEATURES, "coef": np.abs(lr_ref.coef_[0])})
#           .sort_values("coef", ascending=False).head(2)["feature"].tolist())
#
# print(f"Top 2 Árvore:    {top_arvore}")
# print(f"Top 2 Log. Reg.: {top_lr}")
# print(f"Concordam em: {set(top_arvore) & set(top_lr)}")
# print()
# print("(b) A Árvore é mais fácil de explicar passo a passo para uma pessoa leiga")
# print("    ('você é homem E está na 3ª classe, por isso...'), enquanto a Reg.")
# print("    Logística exige entender coeficientes e a função sigmoide.")


**✏️ Minha reflexão sobre a aula:**

1. "Caixa-branca" significa: *...*

2. O maior risco de uma árvore sem limitação de profundidade é: *...*

3. Eu preferiria uma Árvore em vez de um SVM quando: *...*
